In [ ]:
#chatbot that answers questions based on your linkedin profile and summary.
#gradio

#ask another llm to evaluate and answer
#if the answer fails evaluation, then rerun
#agentic workflow, without any framework.

#workflow decided on the output of evaluator llm

#pydantic model, base model
#structured outputs

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
#gradio is a library to create web apps, user interfaces.

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [1]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

NameError: name 'PdfReader' is not defined

In [4]:
print(linkedin)

   
Contact
divyabaheti101@gmail.com
www.linkedin.com/in/divya-baheti
(LinkedIn)
Top Skills
Cascading Style Sheets (CSS)
HTML
React.js
Divya Baheti
Software Engineer at FIS
Pune, Maharashtra, India
Summary
Proficient in developing and maintaining software products for
Windows and web-based applications within the Banking Finance
industry using Agile Methodology like SCRUM, with a specialized
knowledge of IB - Post Trade Services.
Committed to staying up-to-date with the latest developments in
decentralized applications and technologies, with a focus on their
potential for transforming industries.
Experience
FIS
6 years 8 months
Software Engineer - FISCD
August 2022 - Present (3 years 7 months)
Pune, Maharashtra, India
• Created a Web Application that is specifically used by LME (London
• Metal Exchange) to make trades in FISCD system to further process it.
• Designed & developed various Components like a combination of Dropdown,
Text and Date, using Angular Material, and many more. 
• 

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Divya Baheti"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as Divya Baheti. You are answering questions on Divya Baheti's website, particularly questions related to Divya Baheti's career, background, skills and experience. Your responsibility is to represent Divya Baheti for interactions on the website as faithfully as possible. You are given a summary of Divya Baheti's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Divya Baheti. I'ma  software engineer and blockchain and web3 enthusiast. I'm originally from Jalgaon, Maharashtra but I moved to Pune in 2013.\nI love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.\n\n## LinkedIn Profile:\n\xa0 \xa

In [ ]:
#message, the msg user is typing in
#history, the conversation history, comes in openai format
#this is a callback function, it's called when user types in a message and presses enter. Used by gradio.
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

In [ ]:
#pydantic is something which is a framework for specifying a schema or using classes.
#you use a class to describe a particular structure, data struct of info.
#evaluation is a subclass of base model

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [12]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [ ]:
#creates prompt for evaluator llm
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [14]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [ ]:
#evaluate takes in llm reply, the user msg it was replying to and convo history.
#returns an obj of Evaluation class, we defined using pydantic model
#it uses a technique called as structured outputs, for doing this return type.
#structured outputs is a way that u can require a llm to respond in a form of an object, it's just json behind the scenes.
#you use parse, that's the way u call an api to use the struct op 
#instead of chat.completions.create()

def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [16]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [17]:
reply


'As of now, I do not hold a patent. My focus has been primarily on developing software solutions and applications, specifically within the Banking Finance industry, rather than on filing patents. However, I am always interested in exploring innovative ideas and technologies in the blockchain and web3 space, so who knows what the future may hold! If you have any questions about my work or projects, feel free to ask!'

In [20]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The agent correctly states that they do not hold a patent, as there is no information about patents in the provided context. The response also maintains a professional and engaging tone, aligning with the persona, by elaborating on their focus and expressing interest in future innovations, and inviting further questions.')

In [21]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
#testing, to make it fail intentionally.

def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [23]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The agent's response is in Pig Latin, which is highly unprofessional and not engaging for a potential client or employer. The agent should respond in clear, professional English, stating that they do not hold any patents, rather than using informal language that detracts from the persona.
Passed evaluation - returning reply
Passed evaluation - returning reply
